# Pendulum policy and reward heatmap

Trains (or loads from checkpoint) a Pendulum K=10 seed=42 IQ-Learn agent, then plots the learned deterministic policy mu(s) alongside the true Pendulum reward r(s, a=0).

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive/imitation_learning

In [ ]:
!pip install gymnasium -q

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt

from iq_learn import IQLearnAgent, train_iq_learn

os.makedirs('models', exist_ok=True)
os.makedirs('plots', exist_ok=True)

CKPT_PATH = 'models/iqlearn_pendulum_K10_seed42.pt'
EXPERT_PATH = 'expert_data/Pendulum-v1_K10.npz'

In [ ]:
if os.path.exists(CKPT_PATH):
    print('Found checkpoint, skipping training.')
else:
    print('Training Pendulum K=10 seed=42 (about 25 min)...')
    agent, log = train_iq_learn(
        env_name='Pendulum-v1',
        expert_npz_path=EXPERT_PATH,
        seed=42, total_steps=30_000,
        eval_interval=2000, eval_episodes=5,
        hidden_dim=256, batch_size=256,
        lr=1e-4, chi2_coef=0.5, alpha=0.2, auto_alpha=True,
        verbose=True,
    )
    torch.save({
        'actor': agent.actor.state_dict(),
        'critic': agent.critic.state_dict(),
        'critic_target': agent.critic_target.state_dict(),
        'alpha': agent.alpha,
    }, CKPT_PATH)
    print(f'Final={log["eval_reward"][-1]:.1f}  Max={max(log["eval_reward"]):.1f}')

In [ ]:
agent = IQLearnAgent(
    state_dim=3, action_dim=1, discrete=False,
    action_low=np.array([-2.0]), action_high=np.array([2.0]),
    hidden_dim=256, lr=1e-4, alpha=0.2, auto_alpha=True, chi2_coef=0.5,
)
ckpt = torch.load(CKPT_PATH, weights_only=True)
agent.actor.load_state_dict(ckpt['actor'])
agent.critic.load_state_dict(ckpt['critic'])
agent.critic_target.load_state_dict(ckpt['critic_target'])
agent.alpha = ckpt.get('alpha', 0.2)

In [ ]:
N = 100
thetas = np.linspace(-np.pi, np.pi, N)
theta_dots = np.linspace(-8.0, 8.0, N)
T, TD = np.meshgrid(thetas, theta_dots)

obs = np.stack([np.cos(T), np.sin(T), TD], axis=-1).reshape(-1, 3)
obs_t = torch.FloatTensor(obs)

with torch.no_grad():
    a_unit = agent.actor.deterministic(obs_t)
    a_scaled = agent._scale_action(a_unit)
policy_grid = a_scaled.squeeze().numpy().reshape(T.shape)

true_reward_grid = -(T**2 + 0.1 * TD**2)

In [ ]:
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300, 'font.size': 10})

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)

im0 = axes[0].pcolormesh(T, TD, policy_grid, cmap='RdBu_r',
                         shading='auto', vmin=-2, vmax=2)
axes[0].set_title(r'Learned policy $\mu(s)$ (IQ-Learn, K=10)', fontsize=12)
axes[0].set_xlabel(r'$\theta$ (angle, rad)')
axes[0].set_ylabel(r'$\dot\theta$ (angular velocity)')
axes[0].axhline(0, color='k', lw=0.5, alpha=0.5)
axes[0].axvline(0, color='k', lw=0.5, alpha=0.5)
fig.colorbar(im0, ax=axes[0], shrink=0.9, label='torque (N.m)')

im1 = axes[1].pcolormesh(T, TD, true_reward_grid, cmap='viridis', shading='auto')
axes[1].set_title(r'Ground-truth reward $r(s, a{=}0)$', fontsize=12)
axes[1].set_xlabel(r'$\theta$ (angle, rad)')
axes[1].set_ylabel(r'$\dot\theta$ (angular velocity)')
axes[1].axhline(0, color='white', lw=0.5, alpha=0.4)
axes[1].axvline(0, color='white', lw=0.5, alpha=0.4)
axes[1].plot(0, 0, marker='*', color='white', markersize=14, markeredgecolor='black')
fig.colorbar(im1, ax=axes[1], shrink=0.9)

fig.savefig('plots/pendulum_reward_landscape.png')
fig.savefig('plots/pendulum_reward_landscape.pdf')
plt.show()